In [6]:
import pandas as pd

df = pd.read_csv("../data/raw/time_series_60min_singleindex.csv")
df["utc_timestamp"] = pd.to_datetime(df["utc_timestamp"], utc=True)
local_time = df["utc_timestamp"].dt.tz_convert("Europe/Berlin")

df["hour"] = local_time.dt.hour
df["day_of_week"] = local_time.dt.day_of_week
df["is_weekend"] = df["day_of_week"].isin([5, 6])

print(df[["utc_timestamp", "day_of_week", "is_weekend", "hour"]].head(n = 10))

              utc_timestamp  day_of_week  is_weekend  hour
0 2014-12-31 23:00:00+00:00            3       False     0
1 2015-01-01 00:00:00+00:00            3       False     1
2 2015-01-01 01:00:00+00:00            3       False     2
3 2015-01-01 02:00:00+00:00            3       False     3
4 2015-01-01 03:00:00+00:00            3       False     4
5 2015-01-01 04:00:00+00:00            3       False     5
6 2015-01-01 05:00:00+00:00            3       False     6
7 2015-01-01 06:00:00+00:00            3       False     7
8 2015-01-01 07:00:00+00:00            3       False     8
9 2015-01-01 08:00:00+00:00            3       False     9


In [22]:
df["load_lag_24"] = df["DE_load_actual_entsoe_transparency"].shift(24)
df["load_lag_168"] = df["DE_load_actual_entsoe_transparency"].shift(168)

index = 24
print(df["load_lag_24"][index + 1] == df["DE_load_actual_entsoe_transparency"][index - 23])

True


In [26]:
df["rolling_mean_24"] = df["DE_load_actual_entsoe_transparency"].rolling(window = 24).mean()
print(df[["utc_timestamp", "DE_load_actual_entsoe_transparency", "rolling_mean_24"]][20:27])

               utc_timestamp  DE_load_actual_entsoe_transparency  \
20 2015-01-01 19:00:00+00:00                             50313.0   
21 2015-01-01 20:00:00+00:00                             48730.0   
22 2015-01-01 21:00:00+00:00                             48624.0   
23 2015-01-01 22:00:00+00:00                             45668.0   
24 2015-01-01 23:00:00+00:00                             42424.0   
25 2015-01-02 00:00:00+00:00                             40577.0   
26 2015-01-02 01:00:00+00:00                             39707.0   

    rolling_mean_24  
20              NaN  
21              NaN  
22              NaN  
23              NaN  
24     45346.541667  
25     45322.625000  
26     45304.791667  


In [32]:
hours = 24
df["rolling_mean_24_safe"] = df["DE_load_actual_entsoe_transparency"].shift(hours).rolling(window=hours).mean()
print(df[["utc_timestamp", "DE_load_actual_entsoe_transparency", "rolling_mean_24_safe"]][45:55])

               utc_timestamp  DE_load_actual_entsoe_transparency  \
45 2015-01-02 20:00:00+00:00                             53186.0   
46 2015-01-02 21:00:00+00:00                             50691.0   
47 2015-01-02 22:00:00+00:00                             45323.0   
48 2015-01-02 23:00:00+00:00                             40464.0   
49 2015-01-03 00:00:00+00:00                             38338.0   
50 2015-01-03 01:00:00+00:00                             36897.0   
51 2015-01-03 02:00:00+00:00                             36104.0   
52 2015-01-03 03:00:00+00:00                             36453.0   
53 2015-01-03 04:00:00+00:00                             36839.0   
54 2015-01-03 05:00:00+00:00                             38073.0   

    rolling_mean_24_safe  
45                   NaN  
46                   NaN  
47                   NaN  
48          45346.541667  
49          45322.625000  
50          45304.791667  
51          45321.375000  
52          45389.125000  
53      

In [45]:
print(f"Is the data increasingly sorted? {df["utc_timestamp"].is_monotonic_increasing}")
print(f"Amount of duplicate timestamps: {df["utc_timestamp"].duplicated().sum()}")
print(f"Amount of missing hours: {(df["utc_timestamp"].diff().iloc[1:] != pd.Timedelta(hours = 1)).sum()}")
print(f"The average value of the index '48' is the same as manual counted: {df["rolling_mean_24_safe"].iloc[48] - df["DE_load_actual_entsoe_transparency"].iloc[1:25].mean() < 1e-8}")

Is the data increasingly sorted? True
Amount of duplicate timestamps: 0
Amount of missing hours: 0
The average value of the index '48' is the same as manual counted: True
